# Notebook 8: Bayesian Optimization & Optuna

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 3 hours  
**Week 8 — Regularization, Hyperparameter Tuning & Optimization**

---

## Why This Matters

Random search is good. But it still treats each trial as **independent** — it learns nothing from previous evaluations. Bayesian optimization is smarter: after each trial, it updates its understanding of the hyperparameter landscape and uses that knowledge to choose the *next* point wisely. This yields far better results per evaluation, critical when each trial takes minutes or hours.

---

## The Treasure Hunter Analogy

Imagine you're treasure hunting on an unmapped island. The treasure is buried somewhere and depth indicates richness.

- **Random search:** Dig 100 random holes. Some will be deep (good), most shallow (bad). You ignore what each dig tells you about adjacent areas.
- **Bayesian optimization:** Dig in a promising spot. Measure how deep it is. **Update your map** of where treasure might be based on this new information. Choose the NEXT dig spot based on your updated map. Each dig makes you smarter about where to dig next.

This is exactly what Bayesian optimization does: it maintains a probabilistic model of the objective function and uses it to make intelligent, informed decisions about where to evaluate next.

---

## Plain English Explanation

Bayesian optimization has three components:

1. **Surrogate model:** A fast, cheap model of your expensive objective function. After each real evaluation, the surrogate is updated. Common choices: Gaussian Process (GP) or Tree-structured Parzen Estimator (TPE).

2. **Acquisition function:** Tells you WHERE to evaluate next by answering: "Given what we know, which point is most likely to improve on our best result so far?" Common choices: Expected Improvement (EI), Upper Confidence Bound (UCB).

3. **Update loop:** Evaluate objective at the chosen point → update surrogate → repeat.

---

## The Math

**Expected Improvement (EI):**
$$\text{EI}(x) = \mathbb{E}[\max(f(x) - f^*, 0)]$$
where $f^*$ is the current best value. EI is high where:
- The predicted mean $\mu(x)$ is high (exploitation of known good regions), OR
- The uncertainty $\sigma(x)$ is high (exploration of unknown regions)

**TPE (Tree-structured Parzen Estimator) — what Optuna uses:**
- Divide past evaluations into "good" (top 25%) and "bad" (bottom 75%)
- Fit density models $\ell(x) = P(x \mid \text{good})$ and $g(x) = P(x \mid \text{bad})$
- Choose next $x^* = \arg\max \frac{\ell(x)}{g(x)}$ — maximize probability of being good vs bad
- More scalable than GP; handles mixed discrete/continuous spaces well

In [ ]:
# ============================================================
# Cell 1: Imports and Optuna availability check
# Falls back to manual implementation if Optuna not installed
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import ElasticNet, Ridge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from scipy.stats import loguniform, uniform, norm
from scipy.spatial.distance import cdist

np.random.seed(42)

# ---- Check for Optuna ----
OPTUNA_AVAILABLE = False
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_AVAILABLE = True
    print(f"Optuna version {optuna.__version__} is available.")
    print("Will use Optuna for Bayesian optimization.")
except ImportError:
    print("Optuna not installed. Running: pip install optuna")
    import subprocess, sys
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', 'optuna', '-q'],
                            capture_output=True, text=True)
    try:
        import optuna
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        OPTUNA_AVAILABLE = True
        print(f"Optuna installed successfully! Version: {optuna.__version__}")
    except ImportError:
        print("Could not install Optuna. Will use a manual Bayesian optimization simulation.")
        print("All concepts and visualizations are still fully demonstrated.")

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print(f"\nOptuna available: {OPTUNA_AVAILABLE}")

In [ ]:
# ============================================================
# Cell 2: Build synthetic house price dataset
# Same California-style dataset as Notebook 7 for consistency
# ============================================================
np.random.seed(42)

N_SAMPLES   = 500
N_FEATURES  = 20
N_INFORMATIVE = 5

X_raw = np.random.randn(N_SAMPLES, N_FEATURES)
true_coef = np.zeros(N_FEATURES)
true_coef[:N_INFORMATIVE] = [150_000, 80_000, -30_000, 50_000, 120_000]
noise = np.random.randn(N_SAMPLES) * 25_000
y_raw = 450_000 + X_raw @ true_coef + noise

feature_names = (
    ['sq_feet', 'num_rooms', 'age_years', 'distance_cbd', 'school_rating']
    + [f'noise_{i}' for i in range(1, N_FEATURES - N_INFORMATIVE + 1)]
)

split_idx   = int(0.8 * N_SAMPLES)
X_train_raw = X_raw[:split_idx]
X_test_raw  = X_raw[split_idx:]
y_train     = y_raw[:split_idx]
y_test      = y_raw[split_idx:]

scaler      = StandardScaler()
X_train     = scaler.fit_transform(X_train_raw)
X_test      = scaler.transform(X_test_raw)

print(f"Training set : {X_train.shape}")
print(f"Test set     : {X_test.shape}")
print(f"Price range  : ${y_train.min():,.0f} — ${y_train.max():,.0f}")

## Section 1: Gaussian Process — The Surrogate Model

Before building Bayesian optimization, we need to understand the Gaussian Process (GP).  
A GP defines a **probability distribution over functions**. Given a few observations, it estimates:
- $\mu(x)$: the expected function value at any point $x$
- $\sigma(x)$: the uncertainty in that estimate

After each observation, the GP updates both $\mu$ and $\sigma$ — uncertainty collapses near observed points, stays high in unexplored regions.

We'll implement a simple GP from scratch using the **RBF (Radial Basis Function) kernel**:
$$k(x_i, x_j) = \exp\left(-\frac{(x_i - x_j)^2}{2\ell^2}\right)$$

In [ ]:
# ============================================================
# Cell 3: Gaussian Process from scratch
# Implements GP posterior mean and variance for 1D input
# ============================================================
np.random.seed(42)

class SimpleGaussianProcess:
    """
    Minimal Gaussian Process for 1D Bayesian optimization demo.
    Uses RBF kernel: k(xi, xj) = exp(-||xi-xj||^2 / (2*length_scale^2))
    """
    def __init__(self, length_scale=0.3, noise=1e-6):
        self.length_scale = length_scale  # controls smoothness
        self.noise = noise                # numerical stability
        self.X_train = None
        self.y_train = None

    def _rbf_kernel(self, X1, X2):
        """Compute RBF kernel matrix between X1 and X2."""
        # cdist computes pairwise squared Euclidean distances
        sq_dist = cdist(X1.reshape(-1, 1), X2.reshape(-1, 1), 'sqeuclidean')
        return np.exp(-sq_dist / (2 * self.length_scale ** 2))

    def fit(self, X, y):
        """Store training data and compute kernel matrix inverse."""
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        K = self._rbf_kernel(self.X_train, self.X_train)
        K += self.noise * np.eye(len(self.X_train))   # add noise for stability
        self.K_inv = np.linalg.inv(K)
        return self

    def predict(self, X_new):
        """Return posterior mean and standard deviation at X_new."""
        X_new = np.array(X_new)
        K_star  = self._rbf_kernel(X_new, self.X_train)     # (n_new, n_train)
        K_ss    = self._rbf_kernel(X_new, X_new)             # (n_new, n_new)
        # Posterior mean: mu* = K* K^{-1} y
        mu  = K_star @ self.K_inv @ self.y_train
        # Posterior covariance: Sigma* = K** - K* K^{-1} K*^T
        cov = K_ss - K_star @ self.K_inv @ K_star.T
        std = np.sqrt(np.clip(np.diag(cov), 0, None))
        return mu, std

print("SimpleGaussianProcess class defined.")
print("Uses RBF kernel for smooth function estimation.")

In [ ]:
# ============================================================
# Cell 4: Expected Improvement acquisition function
# The heart of Bayesian optimization
# EI(x) = E[max(f(x) - f_best, 0)]
# ============================================================

def expected_improvement(X_candidates, gp, y_best, xi=0.01):
    """
    Compute Expected Improvement at candidate points.

    Parameters
    ----------
    X_candidates : array of points where we might evaluate next
    gp           : fitted SimpleGaussianProcess
    y_best       : best objective value observed so far
    xi           : exploration parameter (higher = more exploration)

    Returns
    -------
    ei : Expected Improvement at each candidate point
    """
    mu, sigma = gp.predict(X_candidates)

    # Avoid division by zero in unexplored regions
    sigma = np.maximum(sigma, 1e-9)

    # Standardized improvement: Z = (mu - f_best - xi) / sigma
    # xi > 0 adds exploration bonus (pushes toward uncertainty)
    Z  = (mu - y_best - xi) / sigma
    ei = (mu - y_best - xi) * norm.cdf(Z) + sigma * norm.pdf(Z)
    ei[sigma < 1e-8] = 0.0  # zero EI where we've already sampled
    return ei

def next_query_point(gp, y_best, bounds, n_candidates=1000):
    """Find the point with highest Expected Improvement."""
    X_candidates = np.linspace(bounds[0], bounds[1], n_candidates)
    ei = expected_improvement(X_candidates, gp, y_best)
    return X_candidates[np.argmax(ei)]

print("Expected Improvement acquisition function defined.")
print()
print("EI balances two things:")
print("  - High mu(x): exploit known good regions")
print("  - High sigma(x): explore uncertain regions")

In [ ]:
# ============================================================
# Cell 5: 1D Bayesian Optimization — step-by-step visualization
# True function: noisy 1D objective (stand-in for CV score)
# Watch the surrogate update after each observation
# ============================================================
np.random.seed(42)

# Define a 1D objective function (pretend this is cross_val_score)
# True function is hidden from the optimizer — it only gets noisy evaluations
def true_objective(x):
    """Synthetic 1D function with multiple peaks — optimizer must find the max."""
    return (np.sin(3 * x) + 0.5 * np.sin(9 * x) + 0.3 * np.cos(5 * x)
            + 0.2 * np.exp(-((x - 0.7) ** 2) / 0.05))  # add a sharp hidden peak

BOUNDS   = (0.0, 1.0)
X_dense  = np.linspace(*BOUNDS, 500)   # grid for plotting the true function
y_true   = true_objective(X_dense)

# Start with 3 random initial observations
X_obs = np.array([0.1, 0.5, 0.9])
y_obs = true_objective(X_obs) + np.random.randn(len(X_obs)) * 0.05

N_ITER       = 8   # number of Bayesian optimization steps to visualize
PLOT_ITERS   = [0, 2, 4, 7]  # which iterations to plot

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes_flat = axes.ravel()

gp = SimpleGaussianProcess(length_scale=0.15, noise=0.01)
X_obs_run = X_obs.copy()
y_obs_run = y_obs.copy()

plot_idx = 0
for iteration in range(N_ITER + 1):
    if iteration in PLOT_ITERS:
        ax = axes_flat[plot_idx]

        # Fit GP on current observations
        gp.fit(X_obs_run, y_obs_run)
        mu, sigma = gp.predict(X_dense)
        ei = expected_improvement(X_dense, gp, y_obs_run.max())

        # True function
        ax.plot(X_dense, y_true, 'k--', linewidth=1.5, alpha=0.5, label='True function (hidden)')

        # GP posterior mean + uncertainty band
        ax.plot(X_dense, mu, 'steelblue', linewidth=2.5, label='GP mean')
        ax.fill_between(X_dense, mu - 2*sigma, mu + 2*sigma,
                        alpha=0.25, color='steelblue', label='±2σ uncertainty')

        # Observed points
        ax.scatter(X_obs_run, y_obs_run, s=80, color='tomato', zorder=5,
                   edgecolors='darkred', linewidths=1.5, label='Observations')

        # Acquisition function (scaled to plot)
        ei_scaled = ei / (ei.max() + 1e-9) * (y_true.max() - y_true.min()) * 0.4 + y_true.min()
        ax.plot(X_dense, ei_scaled, 'mediumseagreen', linewidth=2,
                linestyle=':', label='EI (scaled)')

        # Next query point (maximum EI)
        if iteration < N_ITER:
            x_next = next_query_point(gp, y_obs_run.max(), BOUNDS)
            ax.axvline(x_next, color='gold', linewidth=2.5, linestyle='-.',
                       label=f'Next query: x={x_next:.3f}')

        ax.set_title(f'Iteration {iteration}: {len(X_obs_run)} observations',
                     fontsize=12, fontweight='bold')
        ax.set_xlabel('Hyperparameter value (e.g., alpha)', fontsize=10)
        ax.set_ylabel('Objective (e.g., CV R²)', fontsize=10)
        if plot_idx == 0:
            ax.legend(fontsize=9, loc='lower right')
        plot_idx += 1

    # Add next observation
    if iteration < N_ITER:
        gp.fit(X_obs_run, y_obs_run)
        x_next = next_query_point(gp, y_obs_run.max(), BOUNDS)
        y_next = true_objective(x_next) + np.random.randn() * 0.05
        X_obs_run = np.append(X_obs_run, x_next)
        y_obs_run = np.append(y_obs_run, y_next)

plt.suptitle('Bayesian Optimization in Action: GP Surrogate Updates After Each Observation',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"True maximum at x={X_dense[np.argmax(y_true)]:.3f}, f={y_true.max():.4f}")
print(f"BO found best: x={X_obs_run[np.argmax(y_obs_run)]:.3f}, f={y_obs_run.max():.4f}")

## Section 2: Full Bayesian Optimization Implementation

Now let's build a complete `BayesianOptimizer` class for hyperparameter tuning of our house price model.
This simulates the same behavior as Optuna's TPE when Optuna is not available.

In [ ]:
# ============================================================
# Cell 6: Full Bayesian Optimizer for 2D hyperparameter tuning
# Tunes ElasticNet: alpha (log-scale) and l1_ratio (linear)
# ============================================================
np.random.seed(42)

class BayesianHyperparamOptimizer:
    """
    Bayesian optimization for ML hyperparameter tuning.
    Uses Gaussian Process surrogate + Expected Improvement acquisition.
    Handles log-scale parameters by internally normalizing to [0,1].
    """
    def __init__(self, param_bounds, log_params=None, n_initial=5, random_state=42):
        """
        param_bounds : dict of {'param': (low, high)} — original scale
        log_params   : list of param names to sample on log scale
        n_initial    : random initializations before using GP
        """
        self.param_bounds = param_bounds
        self.log_params   = log_params or []
        self.n_initial    = n_initial
        self.gp           = SimpleGaussianProcess(length_scale=0.2, noise=1e-4)
        self.history      = []   # list of (params_dict, score)
        np.random.seed(random_state)

    def _to_normalized(self, params):
        """Map params to [0,1] space for the GP."""
        x = []
        for name, (lo, hi) in self.param_bounds.items():
            val = params[name]
            if name in self.log_params:
                x.append((np.log10(val) - np.log10(lo)) / (np.log10(hi) - np.log10(lo)))
            else:
                x.append((val - lo) / (hi - lo))
        return np.array(x)

    def _from_normalized(self, x):
        """Map [0,1] space back to original parameter scale."""
        params = {}
        for i, (name, (lo, hi)) in enumerate(self.param_bounds.items()):
            if name in self.log_params:
                params[name] = 10 ** (x[i] * (np.log10(hi) - np.log10(lo)) + np.log10(lo))
            else:
                params[name] = x[i] * (hi - lo) + lo
        return params

    def _random_params(self):
        """Sample a random parameter configuration (used for initialization)."""
        params = {}
        for name, (lo, hi) in self.param_bounds.items():
            if name in self.log_params:
                params[name] = 10 ** np.random.uniform(np.log10(lo), np.log10(hi))
            else:
                params[name] = np.random.uniform(lo, hi)
        return params

    def suggest(self):
        """Return the next params to evaluate."""
        if len(self.history) < self.n_initial:
            return self._random_params()   # random init phase
        # Fit GP on history
        X_hist = np.array([self._to_normalized(p) for p, _ in self.history])
        y_hist = np.array([s for _, s in self.history])
        # For 1D GP demo: use first param only (alpha) as proxy
        # For real: use full X_hist with multi-output GP
        self.gp.fit(X_hist[:, 0], y_hist)
        # Search over a fine grid of alpha values, fix l1_ratio to best seen
        best_p, _ = self.history[np.argmax([s for _, s in self.history])]
        x_candidates = np.linspace(0, 1, 300)
        ei = expected_improvement(x_candidates, self.gp, max(s for _, s in self.history))
        best_alpha_norm = x_candidates[np.argmax(ei)]
        # Random l1_ratio
        l1 = np.random.uniform(*self.param_bounds.get('l1_ratio', (0, 1)))
        x_vec = np.array([best_alpha_norm, (l1 - self.param_bounds['l1_ratio'][0]) /
                           (self.param_bounds['l1_ratio'][1] - self.param_bounds['l1_ratio'][0])])
        return self._from_normalized(x_vec)

    def register(self, params, score):
        """Record the result of an evaluation."""
        self.history.append((params, score))

    @property
    def best_params(self):
        best_idx = np.argmax([s for _, s in self.history])
        return self.history[best_idx][0]

    @property
    def best_score(self):
        return max(s for _, s in self.history)

print("BayesianHyperparamOptimizer class defined.")
print("Combines: GP surrogate + Expected Improvement + normalized parameter space.")

In [ ]:
# ============================================================
# Cell 7: Run custom Bayesian optimization on ElasticNet
# ============================================================
np.random.seed(42)

def evaluate_elasticnet(params, X, y, cv=5):
    """Evaluate ElasticNet with given params using cross-validation."""
    model = ElasticNet(alpha=params['alpha'], l1_ratio=params['l1_ratio'], max_iter=5000)
    scores = cross_val_score(model, X, y, cv=cv, scoring='r2')
    return scores.mean()

# Set up the optimizer
bo = BayesianHyperparamOptimizer(
    param_bounds={'alpha': (1e-3, 100), 'l1_ratio': (0.0, 1.0)},
    log_params=['alpha'],
    n_initial=5,
    random_state=42
)

N_BO_TRIALS = 40
bo_running_best = []

for trial in range(N_BO_TRIALS):
    # Ask optimizer: what should I try next?
    params = bo.suggest()
    # Evaluate the objective (expensive in real life!)
    score = evaluate_elasticnet(params, X_train, y_train)
    # Tell optimizer what we found
    bo.register(params, score)
    bo_running_best.append(bo.best_score)

print("Custom Bayesian Optimization Results:")
print("=" * 45)
print(f"Best alpha    : {bo.best_params['alpha']:.6f}")
print(f"Best l1_ratio : {bo.best_params['l1_ratio']:.6f}")
print(f"Best R²       : {bo.best_score:.6f}")

## Section 3: Optuna — Industrial-Grade Bayesian Optimization

Optuna uses **TPE (Tree-structured Parzen Estimator)** instead of a Gaussian Process.  
TPE is faster, handles mixed discrete/continuous spaces, and scales to high dimensions.

Key Optuna concepts:
- **Trial:** one evaluation of the objective function
- **Study:** the full optimization run (collection of trials)
- **Objective function:** your Python function that returns a score
- **`suggest_float(..., log=True)`:** equivalent to our `loguniform`

In [ ]:
# ============================================================
# Cell 8: Optuna study (or simulated fallback)
# Optimize ElasticNet hyperparameters with Bayesian search
# ============================================================
np.random.seed(42)

N_TRIALS = 80   # number of Bayesian optimization trials

if OPTUNA_AVAILABLE:
    # ---- Real Optuna implementation ----
    def optuna_objective(trial):
        # Suggest hyperparameters — Optuna chooses these intelligently
        alpha    = trial.suggest_float('alpha', 1e-3, 100.0, log=True)
        l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)
        model    = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=5000)
        scores   = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
        return scores.mean()   # Optuna maximizes this

    study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(optuna_objective, n_trials=N_TRIALS, show_progress_bar=False)

    best_params_optuna = study.best_params
    best_score_optuna  = study.best_value

    # Extract per-trial history
    trial_values   = [t.value for t in study.trials]
    optuna_running_best = np.maximum.accumulate(trial_values)

else:
    # ---- Fallback: simulate Bayesian optimization behavior ----
    # Uses our BayesianHyperparamOptimizer to mimic Optuna's behavior
    print("Optuna not available — using simulated Bayesian optimization.")
    print("The behavior is equivalent for demonstration purposes.\n")

    bo_fallback = BayesianHyperparamOptimizer(
        param_bounds={'alpha': (1e-3, 100), 'l1_ratio': (0.0, 1.0)},
        log_params=['alpha'], n_initial=8, random_state=7
    )
    trial_values = []
    for _ in range(N_TRIALS):
        p = bo_fallback.suggest()
        s = evaluate_elasticnet(p, X_train, y_train)
        bo_fallback.register(p, s)
        trial_values.append(s)

    best_params_optuna = bo_fallback.best_params
    best_score_optuna  = bo_fallback.best_score
    # Create a mock study object for compatibility
    study = None
    optuna_running_best = np.maximum.accumulate(trial_values)

print("Bayesian Optimization (Optuna/Custom) Results:")
print("=" * 50)
print(f"Best alpha    : {best_params_optuna['alpha']:.6f}")
print(f"Best l1_ratio : {best_params_optuna['l1_ratio']:.6f}")
print(f"Best R²       : {best_score_optuna:.6f}")

## Section 4: Convergence Comparison — All Three Methods

Now we compare Grid Search, Random Search, and Bayesian Optimization on the same problem with the same evaluation budget. Which method finds the best hyperparameters fastest?

In [ ]:
# ============================================================
# Cell 9: Run grid search and random search for comparison
# ============================================================
np.random.seed(42)

# --- Grid Search (N=64: 8x8 grid) ---
grid_params = {
    'alpha'   : np.logspace(-3, 2, 8),
    'l1_ratio': np.linspace(0, 1, 8)
}
gs = GridSearchCV(ElasticNet(max_iter=5000), grid_params, cv=5, scoring='r2', n_jobs=-1)
gs.fit(X_train, y_train)

# Build grid running-best (deterministic: sorted by order they were evaluated)
cv_results_df = pd.DataFrame(gs.cv_results_)
grid_scores   = cv_results_df['mean_test_score'].values
grid_running_best = np.maximum.accumulate(grid_scores)

# --- Random Search (N=80 trials) ---
rs = RandomizedSearchCV(
    ElasticNet(max_iter=5000),
    {'alpha': loguniform(1e-3, 100), 'l1_ratio': uniform(0, 1)},
    n_iter=80, cv=5, scoring='r2', random_state=42, n_jobs=-1
)
rs.fit(X_train, y_train)
rs_results_df = pd.DataFrame(rs.cv_results_)
rand_scores = rs_results_df['mean_test_score'].values
rand_running_best = np.maximum.accumulate(rand_scores)

print("Grid Search  — best R²:", gs.best_score_)
print("Random Search— best R²:", rs.best_score_)
print("Bayesian Opt — best R²:", best_score_optuna)

In [ ]:
# ============================================================
# Cell 10: Convergence plot — best score vs trial number
# The key comparison: which method is most sample-efficient?
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Truncate all to min length for fair comparison
min_len = min(len(grid_running_best), len(rand_running_best), len(optuna_running_best))
trials  = np.arange(1, min_len + 1)

# --- Left: Convergence curves ---
axes[0].plot(trials, grid_running_best[:min_len],
             color='steelblue', linewidth=2.5, label='Grid Search')
axes[0].plot(trials, rand_running_best[:min_len],
             color='tomato', linewidth=2.5, label='Random Search', linestyle='--')
axes[0].plot(trials, optuna_running_best[:min_len],
             color='mediumseagreen', linewidth=2.5, label='Bayesian Opt (Optuna/Custom)')

# Mark where Bayesian opt first hits 99% of its final best
threshold = 0.99 * optuna_running_best[-1]
first_hit = np.where(optuna_running_best[:min_len] >= threshold)[0]
if len(first_hit) > 0:
    axes[0].axvline(first_hit[0] + 1, color='mediumseagreen', linewidth=1.5,
                    linestyle=':', alpha=0.7, label=f'BO at 99% best: trial {first_hit[0]+1}')

axes[0].set_xlabel('Number of Evaluations', fontsize=12)
axes[0].set_ylabel('Best CV R² Score Found', fontsize=12)
axes[0].set_title('Convergence: Best Score vs. Number of Evaluations',
                  fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)

# --- Right: Trial-by-trial scatter (shows the search pattern) ---
rand_per_trial = pd.DataFrame(rs.cv_results_)['mean_test_score'].values[:min_len]
bo_per_trial   = np.array(trial_values[:min_len])

axes[1].scatter(trials, grid_scores[:min_len], alpha=0.4, s=40,
                color='steelblue', label='Grid: individual trial')
axes[1].scatter(trials, rand_per_trial, alpha=0.4, s=40,
                color='tomato', label='Random: individual trial')
axes[1].scatter(trials, bo_per_trial, alpha=0.5, s=40,
                color='mediumseagreen', label='BO: individual trial')

axes[1].set_xlabel('Trial Number', fontsize=12)
axes[1].set_ylabel('CV R² Score', fontsize=12)
axes[1].set_title('Individual Trial Scores\n(Bayesian Opt should show fewer bad trials over time)',
                  fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

plt.suptitle('Grid Search vs Random Search vs Bayesian Optimization',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 5: Optuna Visualization Suite

If Optuna is available, we use its built-in visualization tools.  
If not, we recreate equivalent plots manually — same information, just built from scratch.

In [ ]:
# ============================================================
# Cell 11: Optimization history plot
# Shows each trial's score and the running best
# ============================================================
if OPTUNA_AVAILABLE and study is not None:
    try:
        fig = optuna.visualization.plot_optimization_history(study)
        fig.show()
    except Exception:
        pass  # fall through to manual version

# Manual version (always runs — either as primary or fallback)
trial_nums   = np.arange(1, len(trial_values) + 1)
running_best = np.maximum.accumulate(trial_values)

fig, ax = plt.subplots(figsize=(13, 6))
ax.scatter(trial_nums, trial_values, alpha=0.5, s=50, color='steelblue',
           label='Trial score', zorder=2)
ax.plot(trial_nums, running_best, color='tomato', linewidth=3,
        label='Best score so far', zorder=3)

# Shade the first n_initial (random) phase
n_init = 8 if not OPTUNA_AVAILABLE else 10
ax.axvspan(0, n_init, alpha=0.1, color='orange', label='Random init phase')
ax.axvspan(n_init, len(trial_values), alpha=0.1, color='mediumseagreen',
           label='Bayesian phase')

ax.set_xlabel('Trial Number', fontsize=12)
ax.set_ylabel('CV R² Score', fontsize=12)
ax.set_title('Bayesian Optimization History: Trial Scores and Running Best',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Print statistics for both phases
init_scores = trial_values[:n_init]
bayes_scores = trial_values[n_init:]
print(f"Random init phase ({n_init} trials): mean={np.mean(init_scores):.4f}, max={np.max(init_scores):.4f}")
print(f"Bayesian phase ({len(bayes_scores)} trials): mean={np.mean(bayes_scores):.4f}, max={np.max(bayes_scores):.4f}")
print("Note: Bayesian phase should have fewer very poor trials.")

In [ ]:
# ============================================================
# Cell 12: Parameter importance analysis
# Which hyperparameter (alpha vs l1_ratio) matters more?
# ============================================================
if OPTUNA_AVAILABLE and study is not None:
    try:
        fig_imp = optuna.visualization.plot_param_importances(study)
        fig_imp.show()
    except Exception:
        pass  # fall through to manual version

# Manual importance analysis via variance in observed scores
# Idea: bin hyperparameter values, compute variance of scores per bin
# High variance -> the parameter has a strong effect on score

# Collect all trials
all_params = [p for p, _ in bo.history] if not OPTUNA_AVAILABLE else [
    {'alpha': t.params['alpha'], 'l1_ratio': t.params['l1_ratio']} for t in study.trials
]
all_scores = [s for _, s in bo.history] if not OPTUNA_AVAILABLE else [
    t.value for t in study.trials
]

alphas   = np.array([p['alpha'] for p in all_params])
l1ratios = np.array([p['l1_ratio'] for p in all_params])
scores   = np.array(all_scores)

# Compute Spearman correlation as a proxy for importance
from scipy.stats import spearmanr
corr_alpha,   p_alpha   = spearmanr(np.log10(alphas), scores)
corr_l1ratio, p_l1ratio = spearmanr(l1ratios, scores)

# Compute variance explained: bin the param and see if score varies between bins
def variance_explained(param_vals, score_vals, n_bins=5):
    bins = np.percentile(param_vals, np.linspace(0, 100, n_bins + 1))
    bin_means = []
    for i in range(n_bins):
        mask = (param_vals >= bins[i]) & (param_vals <= bins[i+1])
        if mask.sum() > 0:
            bin_means.append(score_vals[mask].mean())
    return np.std(bin_means)  # high std = high importance

imp_alpha   = variance_explained(np.log10(alphas), scores)
imp_l1ratio = variance_explained(l1ratios, scores)
total_imp   = imp_alpha + imp_l1ratio + 1e-9

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: importance bar chart
params_names = ['alpha (log-scale)', 'l1_ratio']
importances  = [imp_alpha / total_imp, imp_l1ratio / total_imp]
colors_bar   = ['steelblue', 'tomato']
bars = axes[0].barh(params_names, importances, color=colors_bar, edgecolor='white', height=0.5)
for bar, val in zip(bars, importances):
    axes[0].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.1%}', va='center', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Relative Importance (variance explained)', fontsize=12)
axes[0].set_title('Hyperparameter Importance\n(which param drives score variation?)',
                  fontsize=12, fontweight='bold')
axes[0].set_xlim(0, max(importances) * 1.25)

# Right: score vs alpha (the more important param)
sc = axes[1].scatter(np.log10(alphas), scores, c=l1ratios, cmap='coolwarm',
                     s=70, alpha=0.7, edgecolors='grey', linewidths=0.3)
plt.colorbar(sc, ax=axes[1], label='l1_ratio value')
axes[1].set_xlabel('log10(alpha)', fontsize=12)
axes[1].set_ylabel('CV R² Score', fontsize=12)
axes[1].set_title('Score vs Alpha\n(color = l1_ratio)', fontsize=12, fontweight='bold')

plt.suptitle('Hyperparameter Importance Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Spearman |r| for log10(alpha) vs score : {abs(corr_alpha):.4f}  (p={p_alpha:.4f})")
print(f"Spearman |r| for l1_ratio vs score     : {abs(corr_l1ratio):.4f}  (p={p_l1ratio:.4f})")

In [ ]:
# ============================================================
# Cell 13: 2D contour plot — hyperparameter landscape
# Interpolates all trial results to show the response surface
# ============================================================
if OPTUNA_AVAILABLE and study is not None:
    try:
        fig_ct = optuna.visualization.plot_contour(study, params=['alpha', 'l1_ratio'])
        fig_ct.show()
    except Exception:
        pass  # fall through to manual version

# Manual contour — interpolate scattered trial results onto a grid
from scipy.interpolate import griddata

# Use all available trial data
log_alphas = np.log10(alphas)

# Create interpolation grid
alpha_grid_pts   = np.linspace(log_alphas.min(), log_alphas.max(), 60)
l1ratio_grid_pts = np.linspace(l1ratios.min(), l1ratios.max(), 60)
AA, LL = np.meshgrid(alpha_grid_pts, l1ratio_grid_pts)

# Interpolate scores onto grid
ZZ = griddata(
    np.column_stack([log_alphas, l1ratios]),
    scores,
    (AA, LL),
    method='cubic'
)

fig, ax = plt.subplots(figsize=(10, 7))

# Contour fill
cf = ax.contourf(AA, LL, ZZ, levels=20, cmap='RdYlGn', alpha=0.9)
plt.colorbar(cf, ax=ax, label='CV R² Score')

# Contour lines
ax.contour(AA, LL, ZZ, levels=10, colors='black', alpha=0.3, linewidths=0.8)

# Overlay sampled points
ax.scatter(log_alphas, l1ratios, s=50, c='white', edgecolors='black',
           linewidths=0.8, alpha=0.7, zorder=4, label='Sampled trials')

# Mark best point
best_idx_plot = np.argmax(scores)
ax.scatter(log_alphas[best_idx_plot], l1ratios[best_idx_plot],
           s=250, marker='*', c='gold', edgecolors='black', linewidths=1.5,
           zorder=5, label=f'Best: α={alphas[best_idx_plot]:.4f}, l1={l1ratios[best_idx_plot]:.3f}')

ax.set_xlabel('log10(alpha)  [regularization strength]', fontsize=12)
ax.set_ylabel('l1_ratio  [0=Ridge, 1=Lasso]', fontsize=12)
ax.set_title('Hyperparameter Response Surface\nalpha × l1_ratio → CV R² Score',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 14: Extended Optuna study — Ridge + Polynomial Features
# More hyperparameters: alpha, degree — shows BO in 3-param space
# ============================================================
np.random.seed(42)

# Ridge + Polynomial pipeline
def evaluate_ridge_poly(alpha, degree, X, y, cv=5):
    pipe = Pipeline([
        ('poly',  PolynomialFeatures(degree=int(degree), include_bias=False)),
        ('ridge', Ridge(alpha=alpha))
    ])
    # Use smaller cv to keep it fast for 3-param space
    scores = cross_val_score(pipe, X, y, cv=cv, scoring='r2')
    return scores.mean()

if OPTUNA_AVAILABLE:
    def objective_ridge(trial):
        alpha  = trial.suggest_float('alpha', 1e-3, 1000.0, log=True)
        degree = trial.suggest_int('degree', 1, 3)
        return evaluate_ridge_poly(alpha, degree, X_train, y_train, cv=3)

    study_ridge = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42)
    )
    study_ridge.optimize(objective_ridge, n_trials=50, show_progress_bar=False)
    best_alpha_ridge  = study_ridge.best_params['alpha']
    best_degree_ridge = study_ridge.best_params['degree']
    best_score_ridge  = study_ridge.best_value
else:
    # Manual search over degree and alpha
    best_score_ridge  = -np.inf
    best_alpha_ridge  = None
    best_degree_ridge = None
    bo3 = BayesianHyperparamOptimizer(
        param_bounds={'alpha': (1e-3, 1000), 'degree_cont': (1.0, 3.0)},
        log_params=['alpha'], n_initial=6, random_state=99
    )
    for _ in range(50):
        p = bo3.suggest()
        degree = max(1, min(3, round(p.get('degree_cont', np.random.choice([1, 2, 3])))))
        s = evaluate_ridge_poly(p['alpha'], degree, X_train, y_train, cv=3)
        bo3.register({'alpha': p['alpha'], 'degree_cont': float(degree)}, s)
        if s > best_score_ridge:
            best_score_ridge  = s
            best_alpha_ridge  = p['alpha']
            best_degree_ridge = degree

print("Ridge + Polynomial Features Results:")
print("=" * 45)
print(f"Best alpha  : {best_alpha_ridge:.6f}")
print(f"Best degree : {best_degree_ridge}")
print(f"Best CV R²  : {best_score_ridge:.6f}")

# Evaluate on test set
final_pipe = Pipeline([
    ('poly',  PolynomialFeatures(degree=int(best_degree_ridge), include_bias=False)),
    ('ridge', Ridge(alpha=best_alpha_ridge))
])
final_pipe.fit(X_train, y_train)
y_pred_final = final_pipe.predict(X_test)
print(f"Test R²     : {r2_score(y_test, y_pred_final):.6f}")

In [ ]:
# ============================================================
# Cell 15: Exploration vs Exploitation visualization
# Shows how Bayesian opt balances trying new areas vs refining known good areas
# ============================================================
np.random.seed(42)

# Simulate 30-step BO and track: did each step go near a previous good result?
bo_trace = BayesianHyperparamOptimizer(
    param_bounds={'alpha': (1e-3, 100), 'l1_ratio': (0.0, 1.0)},
    log_params=['alpha'], n_initial=5, random_state=123
)
steps, step_types, step_scores = [], [], []

for i in range(30):
    p = bo_trace.suggest()
    s = evaluate_elasticnet(p, X_train, y_train, cv=3)
    bo_trace.register(p, s)
    steps.append(i)
    step_types.append('Random' if i < 5 else 'Bayesian')
    step_scores.append(s)

# Plot alpha choices over time (log scale)
step_alphas   = [p['alpha'] for p, _ in bo_trace.history]
step_l1ratios = [p['l1_ratio'] for p, _ in bo_trace.history]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: trajectory of alpha choices over trials
colors_trace = ['orange' if t == 'Random' else 'steelblue' for t in step_types]
axes[0].scatter(steps, np.log10(step_alphas), c=step_scores, cmap='RdYlGn',
                s=100, zorder=3, edgecolors='grey', linewidths=0.5)
axes[0].plot(steps, np.log10(step_alphas), alpha=0.3, color='grey', linewidth=1)
for i, t in enumerate(step_types):
    if t == 'Random':
        axes[0].scatter(steps[i], np.log10(step_alphas[i]),
                        s=150, edgecolors='orange', linewidths=2.5, facecolors='none', zorder=4)
axes[0].set_xlabel('Trial Number', fontsize=12)
axes[0].set_ylabel('log10(alpha) chosen', fontsize=12)
axes[0].set_title('Alpha Choices Over Trials\n(orange ring = random init)', fontsize=12)
sm = plt.cm.ScalarMappable(cmap='RdYlGn',
     norm=plt.Normalize(min(step_scores), max(step_scores)))
plt.colorbar(sm, ax=axes[0], label='CV R² Score')

# Right: score trajectory with running best
running_best_trace = np.maximum.accumulate(step_scores)
axes[1].scatter(steps, step_scores, c=colors_trace, s=80, zorder=3, alpha=0.8)
axes[1].plot(steps, running_best_trace, color='tomato', linewidth=2.5,
             label='Best so far')
axes[1].axvspan(-0.5, 4.5, alpha=0.1, color='orange', label='Random phase')
axes[1].axvspan(4.5, 30, alpha=0.05, color='steelblue', label='Bayesian phase')
axes[1].set_xlabel('Trial Number', fontsize=12)
axes[1].set_ylabel('CV R² Score', fontsize=12)
axes[1].set_title('Score Trajectory\nBayesian phase exploits past knowledge', fontsize=12)
axes[1].legend(fontsize=11)

plt.suptitle('Bayesian Optimization: Exploration vs Exploitation Balance',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 16: Final performance summary — all methods on test set
# ============================================================
np.random.seed(42)

# Fit and evaluate best model from each method
methods = {
    'Grid Search'    : {'alpha': gs.best_params_['alpha'],
                        'l1_ratio': gs.best_params_['l1_ratio']},
    'Random Search'  : {'alpha': rs.best_params_['alpha'],
                        'l1_ratio': rs.best_params_['l1_ratio']},
    'Bayesian Opt'   : {'alpha': best_params_optuna['alpha'],
                        'l1_ratio': best_params_optuna['l1_ratio']},
}

test_results = []
for method_name, params in methods.items():
    model = ElasticNet(alpha=params['alpha'], l1_ratio=params['l1_ratio'], max_iter=10000)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    test_results.append({
        'Method'   : method_name,
        'Alpha'    : params['alpha'],
        'l1_ratio' : params['l1_ratio'],
        'Test R²'  : r2_score(y_test, y_pred),
        'Test MAE' : mean_absolute_error(y_test, y_pred),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_pred))
    })

results_tbl = pd.DataFrame(test_results)
print("Final Test Set Performance — All Methods")
print("=" * 80)
print(results_tbl.to_string(index=False))

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 5))
colors_bar2 = ['steelblue', 'tomato', 'mediumseagreen']
bars = ax.bar(results_tbl['Method'], results_tbl['Test R²'],
              color=colors_bar2, edgecolor='white', alpha=0.85)
for bar, val in zip(bars, results_tbl['Test R²']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_ylabel('Test R² Score', fontsize=12)
ax.set_title('Test Performance: Grid vs Random vs Bayesian Optimization',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## Self-Check Questions

Test your understanding. Try answering before looking at the answers.

---

**Question 1:** Bayesian optimization uses an "acquisition function." If you use Expected Improvement (EI), what *exactly* is being maximized at each step?

<details>
<summary>Click to reveal answer</summary>

**Answer:** Expected Improvement maximizes:  
$$\text{EI}(x) = \mathbb{E}[\max(f(x) - f^*, 0)]$$

where $f^*$ is the **best objective value observed so far**.

In plain English: "At candidate point $x$, what is the *expected value of improvement* over what I've already found?"

The key insight is that EI naturally balances exploration and exploitation:
- **Exploitation:** If the surrogate mean $\mu(x)$ is high (above $f^*$), EI is high — we expect improvement.
- **Exploration:** If the surrogate uncertainty $\sigma(x)$ is high (we haven't sampled here), EI is also elevated — there's a non-trivial chance of being much better than $f^*$.

The formula for a Gaussian surrogate evaluates to:  
$$\text{EI}(x) = (\mu(x) - f^* - \xi) \cdot \Phi(Z) + \sigma(x) \cdot \phi(Z)$$
where $Z = (\mu(x) - f^* - \xi) / \sigma(x)$, $\Phi$ is the normal CDF, $\phi$ is the normal PDF, and $\xi \geq 0$ is an exploration parameter.

Each optimization step evaluates the objective function at the $x$ that maximizes EI, then updates the surrogate with the new observation.

</details>

---

**Question 2:** After 50 trials, Bayesian optimization has found a very good region (say $\alpha \approx 0.05$, R² ≈ 0.92). The acquisition function could either exploit (refine near 0.05) or explore (try very different values). What determines which the algorithm chooses?

<details>
<summary>Click to reveal answer</summary>

**Answer:** The balance is controlled by the **uncertainty** in the surrogate model and the **exploration parameter** $\xi$ (or equivalent in UCB: $\kappa$).

After 50 trials near $\alpha \approx 0.05$, the surrogate model has **low uncertainty** in that region — it knows the landscape well there. EI in that region becomes small because there's little chance of beating 0.92 by more than a small amount.

Simultaneously, in regions far from observed points (say $\alpha \approx 50$), the surrogate has **high uncertainty** $\sigma(x)$. EI there could still be high because there's a non-trivial probability of being even better.

**What tips the balance:**
- If $f^*$ is already very high (0.98), the bar for improvement is high → exploitation near the current best becomes the dominant strategy.
- If $f^*$ is moderate (0.75), the chance of finding something much better in unexplored territory is still meaningful → exploration wins.
- The $\xi$ parameter explicitly controls this trade-off: $\xi = 0$ is pure exploitation, $\xi = 0.1$ adds an exploration bonus.

In practice (with Optuna/TPE), this emerges naturally from the ratio $P(x | \text{good}) / P(x | \text{bad})$ — if an unexplored region looks more like the good trials than the bad ones, TPE will direct sampling there.

</details>

---

**Question 3:** Bayesian optimization is inherently sequential — each trial needs the result of the previous one to update the surrogate. This makes it hard to parallelize. How does random search handle parallelism better, and when would you prefer random search for that reason?

<details>
<summary>Click to reveal answer</summary>

**Answer:** Random search is **embarrassingly parallel** — each trial is completely independent of all others. You can run 100 trials simultaneously on 100 machines, completing the entire search in the time it takes for one trial.

Bayesian optimization is **sequential by design**: trial $t+1$ requires the result of trial $t$ to update the surrogate model. In a single-machine setting this is fine, but on a cluster, most machines sit idle waiting for the current trial to finish.

**When to prefer random search:**
1. **You have a large compute cluster** — with 50 machines available, random search can be 50× faster wall-clock time than sequential Bayesian optimization.
2. **Each trial is cheap** — if a CV takes 2 seconds, random search over 200 trials takes 6 minutes. Bayesian optimization takes 6+ minutes just for the sequential overhead.
3. **Many hyperparameters (>10)** — Gaussian Process surrogates become computationally expensive in high dimensions. Random search scales gracefully.

**When to prefer Bayesian:**
1. **Each trial is expensive** (hours of training) — every evaluation saved matters enormously.
2. **Sequential setting** — you're running on one machine with no parallelism available.
3. **Tight budget** (fewer than 50 trials) — Bayesian optimization extracts more signal per trial.

**Note:** Optuna and other modern frameworks support *asynchronous Bayesian optimization* with batch suggestions, partially recovering parallelism while retaining most of the benefit of informed search.

</details>

In [ ]:
# ============================================================
# Cell 17: TPE explanation — how Optuna really works under the hood
# ============================================================
np.random.seed(42)

print("TPE: TREE-STRUCTURED PARZEN ESTIMATOR")
print("=" * 55)
print()
print("Unlike GP-based Bayesian optimization, Optuna's TPE:")
print()
print("1. SPLIT trials into 'good' and 'bad':")
print("   - Good = top gamma fraction (default: top 25%)")
print("   - Bad  = remaining 75%")
print()
print("2. MODEL each group separately:")
print("   - l(x) = P(x | good) — density of x in good trials")
print("   - g(x) = P(x | bad)  — density of x in bad trials")
print()
print("3. CHOOSE next x* = argmax [ l(x) / g(x) ]")
print("   Points where good trials cluster → high l(x)")
print("   Points where bad trials cluster  → high g(x)")
print("   We want x that looks like good trials, not bad ones")
print()
print("ADVANTAGES of TPE over GP:")
print("  - Scales to high dimensions (no matrix inversion)")
print("  - Handles mixed discrete + continuous spaces")
print("  - Handles conditional hyperparameters (param A only")
print("    exists if param B = 'yes')")
print("  - Fast: O(n log n) vs O(n^3) for GP")
print()

# Demonstrate the good/bad split on our data
gamma = 0.25
threshold_percentile = np.percentile(scores, (1 - gamma) * 100)
good_mask = scores >= threshold_percentile
bad_mask  = ~good_mask

print(f"Demonstrating on {len(scores)} trials with gamma={gamma}:")
print(f"  Threshold score (top {int(gamma*100)}%)  : {threshold_percentile:.6f}")
print(f"  'Good' trials                        : {good_mask.sum()}")
print(f"  'Bad'  trials                        : {bad_mask.sum()}")
print(f"  Mean alpha in good trials            : {alphas[good_mask].mean():.6f}")
print(f"  Mean alpha in bad  trials            : {alphas[bad_mask].mean():.6f}")

In [ ]:
# ============================================================
# Cell 18: Visualize good/bad split and TPE density ratio
# This shows exactly how TPE decides where to sample next
# ============================================================
np.random.seed(42)
from scipy.stats import gaussian_kde

log_a = np.log10(alphas)
log_a_good = log_a[good_mask]
log_a_bad  = log_a[bad_mask]

alpha_range = np.linspace(log_a.min() - 0.5, log_a.max() + 0.5, 300)

# Fit KDE density estimators for good and bad
kde_good = gaussian_kde(log_a_good, bw_method=0.4)
kde_bad  = gaussian_kde(log_a_bad,  bw_method=0.4)

l_x = kde_good(alpha_range)
g_x = kde_bad(alpha_range)
ratio = l_x / (g_x + 1e-9)  # TPE selection criterion

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: density of good vs bad trials
axes[0].plot(alpha_range, l_x, color='mediumseagreen', linewidth=2.5,
             label='l(x) = P(x | good trials) — TOP 25%')
axes[0].plot(alpha_range, g_x, color='tomato', linewidth=2.5, linestyle='--',
             label='g(x) = P(x | bad trials) — BOTTOM 75%')
axes[0].scatter(log_a_good, np.zeros_like(log_a_good) - 0.02, s=40,
                color='mediumseagreen', alpha=0.5, zorder=3)
axes[0].scatter(log_a_bad, np.zeros_like(log_a_bad) - 0.04, s=20,
                color='tomato', alpha=0.3, zorder=3)
axes[0].set_xlabel('log10(alpha)', fontsize=12)
axes[0].set_ylabel('Probability Density', fontsize=12)
axes[0].set_title('TPE: Density of Good vs Bad Trials\nWe want x where l(x) is high, g(x) is low',
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)

# Right: the ratio l(x)/g(x) — TPE acquisition function
axes[1].plot(alpha_range, ratio / ratio.max(), color='royalblue', linewidth=3)
axes[1].fill_between(alpha_range, ratio / ratio.max(), alpha=0.2, color='royalblue')
best_next_alpha = alpha_range[np.argmax(ratio)]
axes[1].axvline(best_next_alpha, color='gold', linewidth=2.5, linestyle='--',
                label=f'Next sample: log10(α)={best_next_alpha:.2f} → α={10**best_next_alpha:.4f}')
axes[1].set_xlabel('log10(alpha)', fontsize=12)
axes[1].set_ylabel('l(x)/g(x) — normalized', fontsize=12)
axes[1].set_title('TPE Acquisition Function: l(x)/g(x)\nSample next where this ratio is highest',
                  fontsize=12, fontweight='bold')
axes[1].legend(fontsize=11)

plt.suptitle('How Optuna\'s TPE Decides Where to Sample Next',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 19: Summary — all three methods compared
# Method selection guide for practitioners
# ============================================================
summary_data = {
    'Property': [
        'Algorithm',
        'Sample efficiency',
        'Parallel friendly',
        'Handles continuous params',
        '# Hyperparams it scales to',
        'Implementation complexity',
        'Best for',
    ],
    'Grid Search': [
        'Exhaustive enumeration',
        'Low (wastes evaluations)',
        'Yes (embarrassingly parallel)',
        'No (must discretize)',
        '1–2',
        'Simple',
        'Final fine-tuning of 1–2 params',
    ],
    'Random Search': [
        'Random sampling from distributions',
        'Medium (independent trials)',
        'Yes (embarrassingly parallel)',
        'Yes (continuous distributions)',
        '3–15',
        'Simple',
        'Large clusters, cheap trials',
    ],
    'Bayesian Opt (Optuna)': [
        'GP or TPE surrogate + EI/UCB',
        'High (each trial informs the next)',
        'Limited (sequential by default)',
        'Yes (with log-scale support)',
        '5–30+',
        'Moderate (use Optuna)',
        'Expensive trials (GPU training)',
    ]
}

summary_df = pd.DataFrame(summary_data)
print("HYPERPARAMETER SEARCH METHOD COMPARISON")
print("=" * 100)
for _, row in summary_df.iterrows():
    print(f"\n[{row['Property']}]")
    print(f"  Grid Search       : {row['Grid Search']}")
    print(f"  Random Search     : {row['Random Search']}")
    print(f"  Bayesian Opt      : {row['Bayesian Opt (Optuna)']}")

print()
print("DECISION TREE:")
print("  1–2 params AND fast to evaluate → Grid Search")
print("  3+ params AND large cluster available → Random Search")
print("  Expensive trials (>1 min each) → Bayesian Opt (Optuna)")
print("  Deep learning HPO → Optuna with Hyperband pruner")

In [ ]:
# ============================================================
# Cell 20: Notebook 8 complete summary
# ============================================================
print("=" * 65)
print("NOTEBOOK 8 SUMMARY: BAYESIAN OPTIMIZATION & OPTUNA")
print("=" * 65)
print()
print("WHAT YOU LEARNED:")
print()
print("1. WHY BAYESIAN OPT:")
print("   Random search ignores past results. Bayesian opt learns")
print("   a surrogate model and queries where improvement is likely.")
print()
print("2. THE THREE COMPONENTS:")
print("   a) Surrogate model (GP or TPE): cheap model of objective")
print("   b) Acquisition function (EI/UCB): balances explore/exploit")
print("   c) Update loop: observe → update surrogate → query")
print()
print("3. EXPECTED IMPROVEMENT:")
print("   EI(x) = E[max(f(x) - f_best, 0)]")
print("   High where mean is good OR uncertainty is high")
print()
print("4. OPTUNA / TPE:")
print("   Splits trials into 'good' / 'bad'")
print("   Maximizes l(x)/g(x) — looks like good, unlike bad")
print("   Faster and more scalable than GP for >3 params")
print()
print("5. WHEN TO USE EACH:")
print("   Grid: 1–2 discrete params, fast model")
print("   Random: 3+ params, large cluster, cheap trials")
print("   Bayesian: expensive trials (minutes/hours per eval)")
print()
print("WHAT YOU BUILT:")
print("  - Gaussian Process from scratch (RBF kernel)")
print("  - Expected Improvement acquisition function")
print("  - Step-by-step GP + EI visualization (8 iterations)")
print("  - BayesianHyperparamOptimizer class")
print("  - Optuna study (or simulated fallback)")
print("  - TPE density ratio visualization (l(x)/g(x))")
print("  - 3-method convergence comparison: Grid/Random/Bayes")
print("  - Parameter importance and contour plots")